# DAH 2026 해금팀 — UAV 이상탐지 VAE
**담당:** 광빈 | **환경:** Google Colab + PyTorch  
**탐지 대상:** GPS Spoofing / Altitude Spoofing / Command Injection  
**피처 (8개):** GPS속도 + IMU가속도 + GPS-IMU residual + 고도 + 기압  
**아키텍처:** UGV VAE와 동일 구조 (window=20, latent=16)

In [ ]:
# Step 0. 라이브러리
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import precision_score, recall_score, f1_score

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'디바이스: {device}')

# UAV 피처 정의
# UGV 대비 변경: wheel_vel_l/r → baro_alt, pitch_rate
FEATURE_NAMES = [
    'gps_vel_x',   # GPS 북향 속도 (m/s)
    'gps_vel_y',   # GPS 동향 속도 (m/s)
    'imu_ax',      # IMU 가속도 x
    'imu_ay',      # IMU 가속도 y
    'residual_x',  # GPS속도 - IMU가속도 (정상: ≈0)
    'residual_y',
    'baro_alt',    # 기압계 고도 (m) — UAV 전용
    'pitch_rate',  # 피치 각속도 (rad/s) — UAV 전용
]
WINDOW = 20
N_FEAT = len(FEATURE_NAMES)  # 8
print(f'피처 수: {N_FEAT}')
print(f'피처: {FEATURE_NAMES}')

## Step 1. 합성 데이터 생성
PX4 실제 비행 패턴을 모사한 시뮬레이션 데이터  
- 정상: 순항 비행 (일정 속도 + 안정적 고도 유지)  
- GPS Spoofing: 가짜 GPS 신호 → gps_vel 폭발, IMU 정상 → residual 폭발  
- Altitude Spoofing: 기압계 조작 → baro_alt 급변 (실제 고도와 불일치)  
- Command Injection: 비정상 pitch 명령 주입 → pitch_rate 이상

In [ ]:
def generate_normal(n=1000):
    """정상 UAV 순항 비행: 일정 속도, 안정적 고도"""
    t = np.linspace(0, 10, n)
    gps_vel_x  = 5.0 + np.random.normal(0, 0.05, n)
    gps_vel_y  = np.sin(t * 0.2) * 0.5 + np.random.normal(0, 0.03, n)
    imu_ax     = gps_vel_x + np.random.normal(0, 0.02, n)
    imu_ay     = gps_vel_y + np.random.normal(0, 0.02, n)
    residual_x = gps_vel_x - imu_ax
    residual_y = gps_vel_y - imu_ay
    baro_alt   = 50.0 + np.random.normal(0, 0.1, n)
    pitch_rate = np.random.normal(0, 0.01, n)
    return np.stack([gps_vel_x, gps_vel_y, imu_ax, imu_ay,
                     residual_x, residual_y, baro_alt, pitch_rate], axis=1)

def generate_gps_spoofing(n=300):
    """GPS Spoofing: 앞 1/3 정상, 이후 가짜 신호 지속 주입
    gps_vel 폭발 + residual 폭발 (IMU는 정상 유지)"""
    data  = generate_normal(n)
    start = n // 3
    data[start:, 0] += 12.0
    data[start:, 1] +=  6.0
    data[start:, 4] += 12.0
    data[start:, 5] +=  6.0
    return data

def generate_altitude_spoofing(n=300):
    """Altitude Spoofing: 앞 1/3 정상, 이후 기압계 지속 조작
    baro_alt 급등 (실제 고도/속도 변화 없음)"""
    data  = generate_normal(n)
    start = n // 3
    data[start:, 6] += 25.0
    return data

def generate_command_injection(n=300):
    """Command Injection: 앞 1/3 정상, 이후 비정상 pitch 명령 주입
    pitch_rate + imu 요동"""
    data  = generate_normal(n)
    start = n // 3
    data[start:, 7] = np.random.normal(0, 0.5, n - start)
    data[start:, 2] = np.random.normal(0, 0.5, n - start)
    data[start:, 3] = np.random.normal(0, 0.5, n - start)
    return data

def sliding_window(data, window=20):
    return np.array([data[i:i+window] for i in range(len(data) - window)])

print('데이터 생성 함수 정의 완료')

## Step 2. VAE 모델 정의

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(),
            nn.Linear(128, 64),        nn.ReLU()
        )
        self.fc_mu     = nn.Linear(64, latent_dim)
        self.fc_logvar = nn.Linear(64, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.ReLU(),
            nn.Linear(64, 128),        nn.ReLU(),
            nn.Linear(128, input_dim)
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

def vae_loss(recon, x, mu, logvar, beta=1.0):
    recon_loss = nn.MSELoss()(recon, x)
    kl_loss    = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta * kl_loss

print('VAE 모델 정의 완료')

## Step 3. 데이터 준비 + 학습

In [ ]:
# 데이터 생성
normal     = generate_normal(1000)
gps_spoof  = generate_gps_spoofing(300)
alt_spoof  = generate_altitude_spoofing(300)
cmd_inject = generate_command_injection(300)

# 정규화 (정상 데이터 기준)
mean, std = normal.mean(0), normal.std(0) + 1e-8
def prep(x):
    return sliding_window((x - mean) / std, WINDOW).reshape(-1, WINDOW * N_FEAT)

X_normal     = prep(normal)
X_gps_spoof  = prep(gps_spoof)
X_alt_spoof  = prep(alt_spoof)
X_cmd_inject = prep(cmd_inject)

split   = int(len(X_normal) * 0.8)
X_train = torch.FloatTensor(X_normal[:split])
X_val   = X_normal[split:]
loader  = DataLoader(TensorDataset(X_train), batch_size=32, shuffle=True)

# 모델 초기화
model     = VAE(input_dim=WINDOW * N_FEAT, latent_dim=16).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
print(f'파라미터 수: {sum(p.numel() for p in model.parameters()):,}')

# 학습
history = []
for epoch in range(60):
    model.train()
    total = 0
    for (batch,) in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        recon, mu, logvar = model(batch)
        loss = vae_loss(recon, batch, mu, logvar)
        loss.backward()
        optimizer.step()
        total += loss.item()
    history.append(total / len(loader))
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/60 | Loss: {history[-1]:.4f}')

print('\n학습 완료!')

## Step 4. 이상 점수 계산 + 탐지율

In [ ]:
def anomaly_score(model, X, device='cpu'):
    model.eval()
    with torch.no_grad():
        x = torch.FloatTensor(X).to(device)
        recon, _, _ = model(x)
        return torch.mean((recon - x) ** 2, dim=1).cpu().numpy()

s_normal     = anomaly_score(model, X_val,        device)
s_gps_spoof  = anomaly_score(model, X_gps_spoof,  device)
s_alt_spoof  = anomaly_score(model, X_alt_spoof,  device)
s_cmd_inject = anomaly_score(model, X_cmd_inject, device)

threshold = np.percentile(s_normal, 95)
print(f'임계값 (정상 95th): {threshold:.4f}')
print(f'GPS Spoofing    탐지율: {(s_gps_spoof  > threshold).mean()*100:.1f}%')
print(f'Altitude Spoofing 탐지율: {(s_alt_spoof  > threshold).mean()*100:.1f}%')
print(f'Command Injection 탐지율: {(s_cmd_inject > threshold).mean()*100:.1f}%')

## Step 5. 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history, color='steelblue', linewidth=1.5)
axes[0].set_title('VAE 학습 Loss (UAV)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

axes[1].boxplot(
    [s_normal, s_gps_spoof, s_alt_spoof, s_cmd_inject],
    labels=['정상', 'GPS\nSpoofing', 'Altitude\nSpoofing', 'Command\nInjection']
)
axes[1].axhline(threshold, color='red', linestyle='--',
                label=f'임계값={threshold:.4f}')
axes[1].set_title('UAV 공격 유형별 이상 점수', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Anomaly Score (재구성 오차)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('uav_anomaly_result.png', dpi=150)
plt.show()
print('완료! uav_anomaly_result.png 저장됨')

## Step 6. XAI — 피처별 기여도

In [ ]:
def feature_contribution(model, X, device='cpu'):
    model.eval()
    with torch.no_grad():
        x = torch.FloatTensor(X).to(device)
        recon, _, _ = model(x)
        per_elem = (recon - x) ** 2
        per_elem = per_elem.reshape(-1, WINDOW, N_FEAT)
        return per_elem.mean(dim=1).cpu().numpy()

attack_data = [
    (s_gps_spoof,  X_gps_spoof,  'GPS Spoofing',     'crimson'),
    (s_alt_spoof,  X_alt_spoof,  'Altitude Spoofing', 'darkorange'),
    (s_cmd_inject, X_cmd_inject, 'Command Injection', 'steelblue'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (scores, X_atk, title, color) in zip(axes, attack_data):
    idx  = np.argmax(scores)
    contrib = feature_contribution(model, X_atk[idx:idx+1], device)[0]
    sorted_idx = np.argsort(contrib)[::-1]
    ax.barh([FEATURE_NAMES[i] for i in sorted_idx],
            contrib[sorted_idx], color=color, alpha=0.85)
    ax.set_xlabel('재구성 오차 기여도')
    ax.set_title(f'{title}\n이상 원인 분석')
    ax.invert_yaxis()

plt.suptitle('UAV VAE 이상탐지 XAI — 피처별 기여도', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('uav_xai_contribution.png', dpi=150)
plt.show()
print('XAI 분석 완료!')

## Step 7. 결과 요약표 (보고서용)

In [ ]:
print('=' * 58)
print('UAV 이상탐지 VAE — 결과 요약')
print('=' * 58)
print(f'모델: VAE (latent_dim=16, window=20, feature={N_FEAT})')
print(f'임계값: {threshold:.4f} (정상 95th 퍼센타일)')
print()
print(f'{"공격 유형":<20} {"Precision":>10} {"Recall":>8} {"F1":>8}')
print('-' * 58)

for name, s_atk in [
    ('GPS Spoofing',     s_gps_spoof),
    ('Altitude Spoofing', s_alt_spoof),
    ('Command Injection', s_cmd_inject),
]:
    y_true = np.ones(len(s_atk))
    y_pred = (s_atk > threshold).astype(int)
    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred)
    f = f1_score(y_true, y_pred, zero_division=0)
    print(f'{name:<20} {p:>10.3f} {r:>8.3f} {f:>8.3f}')

print('=' * 58)

In [ ]:
## Step 7-2. 혼동행렬 (Confusion Matrix)
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

configs = [
    ('전체 (All)',          np.concatenate([s_gps_spoof, s_alt_spoof, s_cmd_inject])),
    ('GPS Spoofing',        s_gps_spoof),
    ('Altitude Spoofing',   s_alt_spoof),
    ('Command Injection',   s_cmd_inject),
]

for ax, (name, s_atk) in zip(axes, configs):
    y_true = np.concatenate([np.zeros(len(s_normal)), np.ones(len(s_atk))])
    y_pred = (np.concatenate([s_normal, s_atk]) > threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['정상', '이상'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontweight='bold')

plt.suptitle('혼동행렬 — UAV VAE 이상탐지 분류 성능', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('uav_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: uav_confusion_matrix.png')

In [ ]:
## Step 8. 가중치 저장 (anomaly_detector.py 연동용)
# anomaly_detector.py가 'vae_uav.pth'를 로드함
torch.save(model.state_dict(), 'vae_uav.pth')
print('✅ vae_uav.pth 저장 완료 — anomaly_detector.py (platform=uav) 에서 사용')

# 검증: 저장된 가중치 다시 로드해서 동일 결과 확인
model_loaded = VAE(input_dim=WINDOW * N_FEAT, latent_dim=16)
model_loaded.load_state_dict(torch.load('vae_uav.pth', map_location='cpu'))
model_loaded.eval()
s_check = anomaly_score(model_loaded, X_val, device)
print(f'재로드 검증 — 임계값: {np.percentile(s_check, 95):.4f} (원본과 동일해야 함)')

## Step 9. 시계열 이상 점수 — 공격 발생 시점 시각화
공격이 시작되는 순간 이상 점수가 급등하는 것을 시간 축으로 보여줌  
→ 발표용 핵심 그래프: "VAE가 GPS Spoofing / Altitude Spoofing을 실시간 탐지"

In [ ]:
def timeseries_score(model, data, mean, std, window=20, n_feat=8, device='cpu'):
    normed = (data - mean) / std
    X = sliding_window(normed, window).reshape(-1, window * n_feat)
    return anomaly_score(model, X, device)

def detect_latency(scores, threshold, attack_start_score, timestep_ms=100):
    after = scores[attack_start_score:]
    detected = np.where(after > threshold)[0]
    if len(detected) == 0:
        return None, None
    steps = int(detected[0])
    return steps, steps * timestep_ms

ATTACK_START_SAMPLE = 100
ATTACK_START_SCORE  = ATTACK_START_SAMPLE - WINDOW   # = 80

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=False)

attack_configs = [
    (gps_spoof,   'GPS Spoofing',       'crimson',
     [(100, 300, 'GPS Spoofing 시작')]),
    (alt_spoof,   'Altitude Spoofing',  'darkorange',
     [(100, 300, 'Altitude Spoofing 시작')]),
    (cmd_inject,  'Command Injection',  'steelblue',
     [(100, 300, 'Cmd Injection 시작')]),
]

for ax, (atk_data, title, color, attack_regions) in zip(axes, attack_configs):
    scores = timeseries_score(model, atk_data, mean, std, WINDOW, N_FEAT, device)
    t = np.arange(len(scores))

    ax.plot(t, scores, color=color, linewidth=1.2, label='이상 점수')
    ax.axhline(threshold, color='black', linestyle='--',
               linewidth=1.2, label=f'임계값 {threshold:.4f}')
    ax.fill_between(t, 0, scores,
                    where=(scores > threshold),
                    alpha=0.3, color=color, label='탐지 구간')

    for (x0, x1, label) in attack_regions:
        score_x0 = max(0, x0 - WINDOW)
        score_x1 = min(len(scores), x1 - WINDOW)
        ax.axvspan(score_x0, score_x1, alpha=0.12, color='gray')
        ax.text(score_x0 + 2, scores.max() * 0.75,
                label, fontsize=9, color='dimgray')

    latency_steps, latency_ms = detect_latency(scores, threshold, ATTACK_START_SCORE)
    if latency_steps is not None:
        detect_x = ATTACK_START_SCORE + latency_steps
        ax.annotate(
            f'탐지 지연: {latency_steps}스텝 ({latency_ms}ms)',
            xy=(detect_x, threshold),
            xytext=(detect_x + 10, scores.max() * 0.55),
            fontsize=9, color='darkred', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='darkred', lw=1.2),
        )
        print(f'[{title}] 탐지 지연 = {latency_steps}스텝 ({latency_ms}ms)')
    else:
        print(f'[{title}] 미탐지')

    ax.set_title(f'{title} — 시계열 이상 점수', fontweight='bold')
    ax.set_ylabel('Anomaly Score')
    ax.set_xlabel('타임스텝 (슬라이딩 윈도우 기준, 1스텝=100ms)')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('UAV VAE 이상탐지 — 공격 발생 시점 시각화\n'
             '(임계값 초과 = 탐지, 회색 영역 = 실제 공격 구간)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('uav_timeseries_detection.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n저장: uav_timeseries_detection.png')